# 🏠 AI Interior Designer — Colab Runner
Run all cells top to bottom. Cell 4 prints your public API URL.
Paste it into `frontend/streamlit_app.py` as `API_URL`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q fastapi uvicorn nest_asyncio \
    diffusers transformers accelerate \
    opencv-python-headless Pillow torch torchvision \
    timm einops safetensors xformers python-multipart

# Cloudflare tunnel — no account or token required
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('✅ Dependencies installed.')


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/AI Interior Generator/backend')

In [ ]:
# ── Cell 2: Clone / upload your project files ─────────────────────────────────
# Option A: Clone from GitHub (recommended once you push your code)
# !git clone https://github.com/MuhammadShayan8401/AI-Interior-Designer
# %cd AI-Interior-Designer

# Option B: Upload files manually via the Colab file browser (left panel)
# Make sure you have this structure in Colab:
#   backend/app.py
#   backend/models/
#   backend/routes/
#   backend/utils/

import os
# If you cloned above, uncomment:
# os.chdir('AI-Interior-Designer')
print('Current directory:', os.getcwd())
print('Files:', os.listdir('.'))


In [ ]:
# ── Cell 3: Pre-load all models (do this once to avoid timeout on first request)
import sys
sys.path.insert(0, 'backend')

from models.segmentation import load_segmentation_model
from models.depth import load_depth_model
from models.diffusion import load_diffusion_model

print('Loading all models...')
load_segmentation_model()
load_depth_model()
load_diffusion_model()
print('✅ All models loaded and ready.')


In [ ]:
# ── Cell 4: Start FastAPI server + Cloudflare tunnel ──────────────────────────
import subprocess, threading, time, re, sys
import nest_asyncio
import uvicorn

nest_asyncio.apply()
sys.path.insert(0, 'backend')

from app import app

def start_tunnel():
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    url_re = re.compile(r'https://[\w-]+\.trycloudflare\.com')
    print('Starting Cloudflare tunnel...')
    for line in proc.stdout:
        match = url_re.search(line)
        if match:
            url = match.group(0)
            print('\n' + '=' * 60)
            print(f'  PUBLIC API URL:  {url}')
            print('=' * 60)
            print('>>> Paste this into frontend/streamlit_app.py as API_URL <<<')
            break

thread = threading.Thread(target=start_tunnel, daemon=True)
thread.start()
time.sleep(3)

uvicorn.run(app, host='0.0.0.0', port=8000)
